# 29. Biomarker Scoring Test (Pipeline 10)

- Goal: verify ⑩ Biomarker Scoring converts ⑧ feature and ⑨ biomech records into biomarker records, saves follow-along outputs, and withholds unavailable score evidence.
- Prerequisites: notebooks 20-28, or equivalent pipeline outputs through ⑨.
- Inputs: p01 squat pose CSV, annotation CSV, promoted `squat` exercise definition, optional baseline JSON.
- Outputs: `data/processed/biomarker/<recording_id>_biomarkers.csv`, `_biomarker_scores.csv`, `_biomarker_score_items.csv`, and `_biomarker_qc.json`.
- Validation points: biomarker provenance exists, score rows follow explicit baseline availability, low-confidence biomech evidence is attenuated or withheld by policy, and pipeline integration matches the direct call contract.

See `docs_eng/pipeline/10_biomarker_scoring.md` and `docs/pipeline/10_biomarker_scoring.md`.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import json
import warnings

import pandas as pd
from IPython.display import display

from movement.biomech import extract_rep_biomech
from movement.biomarker import (
    BiomarkerRecord,
    save_biomarker_outputs,
    score_records_to_item_dataframe,
)
from movement.biomarker.scoring import (
    BiomarkerScoreRecord,
    derive_biomarkers,
    load_baseline,
)
from movement.core.io import load_pose_csv
from movement.definitions.exercise_definition import load_exercise_definition
from movement.features import extract_rep_features, summarize_phase_to_rep
from movement.pipeline import load_pipeline_config, run_pipeline
from movement.stage_context import (
    build_stage_check_pipeline_config,
    find_project_root,
    recording_id_from_pose_csv,
    resolve_target_definitions_dir,
)

warnings.filterwarnings("ignore", category=RuntimeWarning)


## Setup

Use the promoted runtime `squat` definition. Authoring draft bundles remain examples in `notebook/10_manual_preparation` and are not the default follow-along target here.


In [ ]:
PROJECT_ROOT = find_project_root()
POSE_CSV = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv"
ANNOTATION_CSV = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv"
TARGET_EXERCISE_ID = "squat"
DEFINITIONS_DIR = resolve_target_definitions_dir(
    TARGET_EXERCISE_ID,
    project_root=PROJECT_ROOT,
)
BASELINE_PATH = PROJECT_ROOT / "data/reference/baseline_zscore.json"
DEFAULT_PIPELINE_CONFIG = load_pipeline_config(PROJECT_ROOT / "configs/pipeline_default.yaml")
BIOMARKER_OUTPUT_DIR = PROJECT_ROOT / "data/processed/biomarker"
RECORDING_ID = recording_id_from_pose_csv(POSE_CSV)

raw_df = load_pose_csv(POSE_CSV)
exercise_def = load_exercise_definition(TARGET_EXERCISE_ID, DEFINITIONS_DIR)
baseline_entry = load_baseline(BASELINE_PATH, TARGET_EXERCISE_ID)

display(
    pd.DataFrame(
        [
            {
                "recording_id": RECORDING_ID,
                "exercise_id": exercise_def.exercise_id,
                "definition_dir": str(DEFINITIONS_DIR.relative_to(PROJECT_ROOT)),
                "pose_rows": len(raw_df),
                "baseline_entry_available": bool(baseline_entry),
            }
        ]
    )
)

assert exercise_def.exercise_id == TARGET_EXERCISE_ID
assert POSE_CSV.exists()
assert ANNOTATION_CSV.exists()


## Upstream Pipeline Through ⑨

Run stages ①-⑨ with the same stage-check helper used by the previous notebooks. ⑩ is disabled here so the direct scoring call below can be inspected separately.


In [ ]:
upstream_cfg = build_stage_check_pipeline_config(
    exercise_id=TARGET_EXERCISE_ID,
    definitions_dir=DEFINITIONS_DIR,
    annotation_csv=ANNOTATION_CSV,
    enable_validation=True,
    enable_annotation=True,
    enable_preprocessing=True,
    enable_normalization=True,
    enable_canonicalization=True,
    enable_rep_segmentation=True,
    enable_phase_segmentation=True,
    enable_features=True,
    enable_role_context=True,
    enable_biomech=True,
)

upstream_df, upstream_report = run_pipeline(raw_df, upstream_cfg)

stage_rows = []
for key in [
    "validation",
    "annotation",
    "exercise_definition",
    "preprocessing",
    "normalization",
    "canonicalization",
    "rep_segmentation",
    "phase_segmentation",
    "features",
    "biomech",
]:
    value = upstream_report.get(key)
    stage_rows.append(
        {
            "stage": key,
            "available": value is not None,
            "rows": len(value) if isinstance(value, list) else None,
        }
    )
display(pd.DataFrame(stage_rows))

assert "features" in upstream_report
assert "biomech" in upstream_report
assert len(upstream_report["features"]) > 0
assert len(upstream_report["biomech"]) > 0


## Direct Biomarker Derivation

Rebuild record objects from the upstream dataframe, then call `derive_biomarkers`. The promoted `squat` path should produce score rows when its baseline entry exists; newly authored exercises without a baseline still produce biomarker rows with zero score rows.


In [ ]:
feature_records = extract_rep_features(upstream_df, exercise_def)
phase_summary_records = summarize_phase_to_rep(feature_records, exercise_def)
feature_records = feature_records + phase_summary_records
biomech_records = extract_rep_biomech(
    upstream_df,
    exercise_def,
    use_visibility_weight=True,
)

with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always", UserWarning)
    biomarker_records, score_records = derive_biomarkers(
        feat_records=feature_records,
        biomech_records=biomech_records,
        exercise_definition=exercise_def,
        definition_version=exercise_def.version,
        baseline_path=BASELINE_PATH,
        domain_weights=DEFAULT_PIPELINE_CONFIG.biomarker.domain_weights,
        low_confidence_score_weights=DEFAULT_PIPELINE_CONFIG.biomarker.low_confidence_score_weights,
        depth_dependency_score_weights=DEFAULT_PIPELINE_CONFIG.biomarker.depth_dependency_score_weights,
        feature_score_weight_overrides=DEFAULT_PIPELINE_CONFIG.biomarker.feature_score_weight_overrides,
        domain_feature_family_weights=DEFAULT_PIPELINE_CONFIG.biomarker.domain_feature_family_weights,
    )

warning_messages = [str(item.message) for item in caught_warnings]
display(
    pd.DataFrame(
        [
            {
                "feature_records": len(feature_records),
                "biomech_records": len(biomech_records),
                "biomarker_records": len(biomarker_records),
                "score_records": len(score_records),
                "baseline_entry_available": bool(baseline_entry),
                "warnings": len(warning_messages),
            }
        ]
    )
)
for message in warning_messages:
    print(message)

assert len(biomarker_records) == len(feature_records) + len(biomech_records)
if baseline_entry:
    assert len(score_records) > 0
else:
    assert len(score_records) == 0


## Check 1: Biomarker Provenance

Every biomarker row must keep exercise identity, unit, value, and source-field provenance. Availability is metadata for later validation/scoring policy, not a hidden correction.


In [ ]:
for record in biomarker_records:
    assert isinstance(record, BiomarkerRecord)
    assert record.exercise_id == TARGET_EXERCISE_ID
    assert record.biomarker_id
    assert record.unit
    assert record.value is not None
    assert record.source_fields

biomarker_df = pd.DataFrame([record.as_dict() for record in biomarker_records])
display(
    biomarker_df.groupby(["availability"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
)
display(biomarker_df.head(8))


## Check 2: Saved Biomarker Outputs

Save the stage-check artifacts. `_biomarker_scores.csv` contains one row per scored rep when the active exercise has a baseline entry. `_biomarker_score_items.csv` expands each scored feature into a readable item-level score audit. If no baseline exists, both score tables remain zero-row CSVs with the current schemas.


In [ ]:
saved_outputs = save_biomarker_outputs(
    biomarker_records=biomarker_records,
    score_records=score_records,
    recording_id=RECORDING_ID,
    exercise_id=TARGET_EXERCISE_ID,
    output_dir=BIOMARKER_OUTPUT_DIR,
    project_root=PROJECT_ROOT,
)
display(saved_outputs)

biomarker_csv = BIOMARKER_OUTPUT_DIR / f"{RECORDING_ID}_biomarkers.csv"
score_csv = BIOMARKER_OUTPUT_DIR / f"{RECORDING_ID}_biomarker_scores.csv"
score_item_csv = BIOMARKER_OUTPUT_DIR / f"{RECORDING_ID}_biomarker_score_items.csv"
qc_json = BIOMARKER_OUTPUT_DIR / f"{RECORDING_ID}_biomarker_qc.json"

saved_biomarker_df = pd.read_csv(biomarker_csv)
saved_score_df = pd.read_csv(score_csv)
saved_score_item_df = pd.read_csv(score_item_csv)
qc_payload = json.loads(qc_json.read_text(encoding="utf-8"))
expected_score_items = score_records_to_item_dataframe(score_records)

assert len(saved_biomarker_df) == len(biomarker_records)
assert len(saved_score_df) == len(score_records)
assert len(saved_score_item_df) == len(expected_score_items)
assert qc_payload["score_available"] == bool(score_records)
assert qc_payload["score_item_rows"] == len(saved_score_item_df)

score_item_columns = [
    "rep_id",
    "domain",
    "feature_id",
    "item_score",
    "deduction",
    "value",
    "z",
    "confidence_weight",
    "depth_dependency",
    "focus_tier",
]
if not saved_score_item_df.empty:
    display(
        saved_score_item_df.sort_values("deduction", ascending=False)[
            score_item_columns
        ].head(12)
    )
display(pd.DataFrame([qc_payload]))


## Check 4: Pipeline Integration

Enable ⑩ inside `run_pipeline`. The report should contain pass-through biomarker records; `biomarker_scores` appears only when a baseline entry exists for the active exercise.


In [ ]:
pipeline_cfg = build_stage_check_pipeline_config(
    exercise_id=TARGET_EXERCISE_ID,
    definitions_dir=DEFINITIONS_DIR,
    annotation_csv=ANNOTATION_CSV,
    enable_validation=True,
    enable_annotation=True,
    enable_preprocessing=True,
    enable_normalization=True,
    enable_canonicalization=True,
    enable_rep_segmentation=True,
    enable_phase_segmentation=True,
    enable_features=True,
    enable_role_context=True,
    enable_biomech=True,
    enable_biomarker=True,
)
pipeline_cfg.biomarker = DEFAULT_PIPELINE_CONFIG.biomarker
pipeline_cfg.biomarker.enabled = True

with warnings.catch_warnings(record=True) as pipeline_warnings:
    warnings.simplefilter("always", UserWarning)
    pipeline_df, pipeline_report = run_pipeline(raw_df, pipeline_cfg)

pipeline_warning_messages = [str(item.message) for item in pipeline_warnings]
pipeline_biomarkers = pipeline_report.get("biomarker", [])
pipeline_scores = pipeline_report.get("biomarker_scores", [])

display(
    pd.DataFrame(
        [
            {
                "pipeline_rows": len(pipeline_df),
                "pipeline_biomarkers": len(pipeline_biomarkers),
                "pipeline_scores": len(pipeline_scores),
                "pipeline_warnings": len(pipeline_warning_messages),
                "baseline_entry_available": bool(baseline_entry),
            }
        ]
    )
)
for message in pipeline_warning_messages:
    print(message)

assert len(pipeline_biomarkers) > 0
if baseline_entry:
    assert len(pipeline_scores) > 0
else:
    assert len(pipeline_scores) == 0


## Check Summary

This notebook confirms that ⑩ produces traceable biomarker rows for the promoted exercise, saves the current output contract, and keeps composite scoring dependent on explicit baseline availability and feature availability metadata.


In [ ]:
check_summary = pd.DataFrame(
    [
        {
            "check": "squat biomarker records",
            "status": "PASS",
            "detail": f"{len(biomarker_records)} records",
        },
        {
            "check": "squat score availability",
            "status": "PASS",
            "detail": (
                f"{len(score_records)} score records; "
                f"baseline_entry_available={bool(baseline_entry)}"
            ),
        },
        {
            "check": "item-level score audit",
            "status": "PASS",
            "detail": f"{len(saved_score_item_df)} scored feature rows",
        },
        {
            "check": "saved output contract",
            "status": "PASS",
            "detail": str(BIOMARKER_OUTPUT_DIR.relative_to(PROJECT_ROOT)),
        },
        {
            "check": "pipeline integration",
            "status": "PASS",
            "detail": f"{len(pipeline_biomarkers)} pipeline biomarker rows",
        },
    ]
)
display(check_summary)
